In [ ]:
# @title 1. 환경 설정 및 라이브러리 설치
"""
이 셀에서는 실습에 필요한 라이브러리들을 설치하고 임포트한다.
- transformers: Hugging Face 모델 및 파이프라인 사용을 위한 라이브러리
- datasets: Hugging Face datasets 라이브러리 (Dataset 객체 생성용)
- sentencepiece: NLLB 등 일부 모델에서 사용하는 토크나이저 라이브러리
- accelerate: 모델 로딩 및 분산 처리를 도와주는 라이브러리
- torch: PyTorch 라이브러리 (기본 백엔드)
- matplotlib / seaborn: 결과 시각화용
"""
# 필요한 라이브러리를 설치
!pip install -q "transformers<5.0" "tokenizers<0.21" datasets sentencepiece accelerate torch matplotlib seaborn

import datasets
from datasets import Dataset
from transformers import pipeline
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# GPU 사용 가능 여부 확인 및 설정
device = 0 if torch.cuda.is_available() else -1
print(f"사용 가능한 디바이스: {'GPU' if device == 0 else 'CPU'}")

사용 가능한 디바이스: CPU


In [ ]:
# @title 2. 데이터셋 로드 및 준비
"""
이 셀에서는 K-pop 가사 데이터셋을 GitHub에서 가져와 로드함.
데이터셋: EX3exp/Kpop-lyric-datasets (멜론 월간차트 기반 2000~2023년 K-pop 곡)
각 JSON 파일이 한 곡이며, song_name / artist / genre / lyrics.lines 등이 들어있음.
전체 데이터셋이 크기 때문에 일부만 사용할 예정
'lyrics' 컬럼이 우리가 영어 번역/요약/감정분석할 원본 가사 -> 이후 1절로 자를 예정임.
"""
import os, glob, json, random

# GitHub에서 K-pop 가사 데이터셋 clone (이미 받아져 있으면 skip)
if not os.path.exists("Kpop-lyric-datasets"):
    !git clone -q https://github.com/EX3exp/Kpop-lyric-datasets.git

# JSON 파일 경로 전부 수집
json_paths = glob.glob("Kpop-lyric-datasets/melon/monthly-chart/**/*.json", recursive=True)
print(f"전체 JSON 파일 수: {len(json_paths)}")

# JSON을 파싱해서 필요한 필드만 추출
def load_song(path):
    with open(path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    lines = d.get('lyrics', {}).get('lines', [])
    # 빈 줄 제거 후 \n으로 합치기
    lyrics_text = "\n".join([ln for ln in lines if ln.strip()])
    return {
        "song_name": d.get("song_name", ""),
        "artist": d.get("artist", ""),
        "genre": d.get("genre", ""),
        "release_date": d.get("release_date", ""),
        "lyrics": lyrics_text,
    }

# 재현성을 위해 시드 고정 후 일부만 샘플링
random.seed(42) # 어느 환경에서나 동일한 곡들 추출
random.shuffle(json_paths) #다양한 장르, 년도를 섞어 사용하기 위해서

# 실습을 위해 데이터 일부만 선택
num_samples_to_use = 10

records = []
for p in json_paths:
    try:
        rec = load_song(p)
        # 가사가 너무 짧거나 비어있는 곡은 제외
        if len(rec["lyrics"]) < 50:
            continue
        # 중복 라인 제거(후렴 반복 줄이기 위해) - 순서 유지하면서 중복 제거
        seen, dedup = set(), []
        for ln in rec["lyrics"].split("\n"):
            if ln not in seen:
                seen.add(ln)
                dedup.append(ln)
        rec["lyrics"] = "\n".join(dedup)
        records.append(rec)
        if len(records) >= num_samples_to_use:
            break
    except Exception as e:
        continue

# Hugging Face Dataset 객체로 변환
kpop_subset = Dataset.from_pandas(pd.DataFrame(records))

print("로드된 데이터셋 정보:")
print(kpop_subset)

print("\n첫 번째 데이터 예시:")
print(f"곡명: {kpop_subset[0]['song_name']}")
print(f"아티스트: {kpop_subset[0]['artist']}")
print(f"장르: {kpop_subset[0]['genre']}")
print(f"가사 (앞부분):\n{kpop_subset[0]['lyrics'][:200]}...")

# 데이터셋 확인을 위해 Pandas DataFrame으로 변환
df_check = pd.DataFrame(kpop_subset)
print("\n데이터셋 일부 미리보기 (DataFrame):")
display(df_check[['song_name', 'artist', 'genre']].head(5))

전체 JSON 파일 수: 25876
로드된 데이터셋 정보:
Dataset({
    features: ['song_name', 'artist', 'genre', 'release_date', 'lyrics'],
    num_rows: 10
})

첫 번째 데이터 예시:
곡명: FOREVER 1
아티스트: 소녀시대 (GIRLS' GENERATION)
장르: 댄스
가사 (앞부분):
FOREVER 1
It’s love It’s love
We’re not stopping
네가 머문 이 세상이 더
아름다운 건
겁 없이 외치던 말 '사랑해 너를'
영원하기에
You and I
터지는 눈물이
말하잖아
난 그냥 전부 던진 거야
아무런 망설임 따위도
멋대로 끌렸던 그대로
Oh my baby 달려가 안을게
I love 너의 모든 것, 내 전부인 너
...

데이터셋 일부 미리보기 (DataFrame):


,song_name,artist,genre
0,FOREVER 1,소녀시대 (GIRLS' GENERATION),댄스
1,나야 나 (PICK ME),PRODUCE 101,댄스
2,악몽,버블 시스터즈,댄스
3,Love,브라운아이드걸스,댄스
4,Young Gunz,신화,R&B/Soul


In [ ]:
# @title 3. 번역 모델 파이프라인 로드
"""
이 셀에서는 한국어 가사를 영어로 번역하기 위한 NLLB 모델 파이프라인을 로드함.
모델: facebook/nllb-200-distilled-600M
파이프라인 타입: translation
NLLB 모델은 다양한 언어를 지원하며, 언어 코드를 지정해야 한다.
한국어: kor_Hang, 영어: eng_Latn

한국어 요약 모델(KoBART)이 가사처럼 영어/한국어가 섞인 텍스트를 잘 못 다뤄서 결과가 깨짐.
그래서 순서를 뒤집어 '한국어 가사 → 영어 가사 → 영어 요약 → 감정 분석'으로 진행함.
"""
# 'translation' 파이프라인을 로드하고, 사용할 모델은 'facebook/nllb-200-distilled-600M'로 지정
# GPU 사용 설정(device=device)도 추가
translator = pipeline(
    task="translation",
    model="facebook/nllb-200-distilled-600M",
    device=device
)

print("번역 파이프라인 로드 완료.")

# 번역 테스트
test_translation = translator("오늘 너무 행복해", src_lang="kor_Hang", tgt_lang="eng_Latn")
print(f"번역 테스트: {test_translation}")

번역 파이프라인 로드 완료.
번역 테스트: [{'translation_text': "I'm so happy today."}]


In [ ]:
# @title 4. 가사 영어로 번역 및 데이터셋에 추가
"""
이 셀에서는 로드된 번역 파이프라인을 사용하여 데이터셋의 'lyrics' 내용을 영어로 번역함.
가사는 한 곡 전체가 길어서 NLLB 토큰 제한에 걸릴 수 있으므로, 라인별로 끊어서 번역한 뒤 다시 합치는 방식을 사용함.
이미 영어로 된 라인(예: "You and I", "I love you")은 그대로 보존된다.
map 함수를 사용하여 데이터셋의 각 샘플에 번역 함수를 적용하고,
결과를 'english_lyrics'라는 새로운 컬럼에 저장함.
"""
import re

def translate_lyrics_to_english(example):
    """데이터셋의 'lyrics'를 받아 1절(앞 20줄)만 영어 번역하는 함수(효율성을 위해)"""
    lines = [ln.strip() for ln in example['lyrics'].split('\n') if ln.strip()]
    lines = lines[:20]   # ← 앞 20줄

    translated_lines = []
    for ln in lines:
        # 한글이 없는 라인(이미 영어)은 번역하지 않고 그대로 사용
        if not re.search(r'[가-힣]', ln):
            translated_lines.append(ln)
            continue
        try:
            result = translator(
                ln,
                src_lang="kor_Hang",
                tgt_lang="eng_Latn",
                max_length=128
            )
            translated_lines.append(result[0]['translation_text'])
        except Exception:
            translated_lines.append(ln)

    example['english_verse'] = "\n".join(translated_lines)
    return example

print("번역 작업을 시작합니다... (1절만 번역하므로 빠르게 진행됩니다)")
translated_dataset = kpop_subset.map(translate_lyrics_to_english)
print("번역 작업 완료.")

print("\n번역이 추가된 데이터셋 정보:")
print(translated_dataset)

print("\n첫 번째 데이터의 한국어 가사와 영어 번역 비교:")
print(f"--- 곡: {translated_dataset[0]['song_name']} / {translated_dataset[0]['artist']} ---")
print("\n--- 한국어 가사 (앞부분) ---")
print(translated_dataset[0]['lyrics'][:300])
print("\n--- 영어 1절 번역 ---")
print(translated_dataset[0]['english_verse'])

# 데이터셋 확인
df_check_translation = pd.DataFrame(translated_dataset)
print("\n번역 추가 후 데이터셋 미리보기 (DataFrame):")
display(df_check_translation[['song_name', 'artist', 'genre', 'english_verse']].head(5))


번역 작업을 시작합니다... (1절만 번역하므로 빠르게 진행됩니다)


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

번역 작업 완료.

번역이 추가된 데이터셋 정보:
Dataset({
    features: ['song_name', 'artist', 'genre', 'release_date', 'lyrics', 'english_verse'],
    num_rows: 10
})

첫 번째 데이터의 한국어 가사와 영어 번역 비교:
--- 곡: FOREVER 1 / 소녀시대 (GIRLS' GENERATION) ---

--- 한국어 가사 (앞부분) ---
FOREVER 1
It’s love It’s love
We’re not stopping
네가 머문 이 세상이 더
아름다운 건
겁 없이 외치던 말 '사랑해 너를'
영원하기에
You and I
터지는 눈물이
말하잖아
난 그냥 전부 던진 거야
아무런 망설임 따위도
멋대로 끌렸던 그대로
Oh my baby 달려가 안을게
I love 너의 모든 것, 내 전부인 너
우리는 영원
We are one
전율 속에 뜨거운 그 맘을 던져
Just like a love bomb
Girls, We are forever
(Yeah we are, We’re 

--- 영어 1절 번역 ---
FOREVER 1
It’s love It’s love
We’re not stopping
You're the only one in the world who can't stop me.
It's beautiful.
I was afraid to say, "I love you".
For eternity,
You and I
The tears that burst
You know what?
I just threw it all away.
I'm not hesitant to say anything.
Just like I was attracted to.
Oh, my baby, I'm not running.
I love you, you're all that I have.
We are eternal.
We are one
Throw that hot heart into the electric

,song_name,artist,genre,english_verse
0,FOREVER 1,소녀시대 (GIRLS' GENERATION),댄스,FOREVER 1\nIt’s love It’s love\nWe’re not stop...
1,나야 나 (PICK ME),PRODUCE 101,댄스,The moment I saw you.\nPick me Pick me Pick me...
2,악몽,버블 시스터즈,댄스,I woke up in the morning with a nightmare and ...
3,Love,브라운아이드걸스,댄스,"My love for you, you little bitch.\nI need you..."
4,Young Gunz,신화,R&B/Soul,NewYork to Seoul This is\nThe Rock Dizzle C`MO...


In [ ]:
# @title 5. 요약 모델 파이프라인 로드
"""
이 셀에서는 영어로 번역된 가사를 요약하기 위한 BART 모델 파이프라인을 로드함.
모델: facebook/bart-large-cnn
파이프라인 타입: summarization
"""
# 'summarization' 파이프라인을 로드하고, 사용할 모델은 'facebook/bart-large-cnn'로 지정함.
# GPU 사용 설정(device=device)도 추가
summarizer = pipeline(
    task="summarization",
    model="facebook/bart-large-cnn",
    device=device
)

print("요약 파이프라인 로드 완료.")

요약 파이프라인 로드 완료.


In [ ]:
# @title 6. 영어 가사 요약 및 데이터셋에 추가
"""
이 셀에서는 번역된 'english_lyrics' 컬럼을 요약함.
가사 특성상 같은 표현이 반복되기 쉬우므로, 반복 억제 옵션(no_repeat_ngram_size,
repetition_penalty)을 추가해 불필요한 데이터를 제거한다.
map 함수를 사용하여 요약 함수를 적용하고, 결과를 'summary' 컬럼에 저장함.
"""
def summarize_english_lyrics(example):
    """데이터셋의 'english_lyrics'를 받아 영어로 요약하는 함수"""
    #summarizer 파이프라인을 사용하여 example['english_lyrics']를 요약한다.
    # 요약 최대 길이는 80, 최소 길이는 20으로 설정
    summary_result = summarizer(
        example['english_verse'],
        max_length=80,
        min_length=20,
        do_sample=False,
        truncation=True,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
        num_beams=4
    )
    # 파이프라인 결과에서 실제 요약 텍스트를 추출하여 example 딕셔너리의 'summary' 키 값으로 저장함.
    example['summary'] = summary_result[0]['summary_text']
    return example

# translated_dataset 데이터셋의 map 함수를 사용하여 위에서 정의한 summarize_english_lyrics 함수를 적용
# 결과를 summarized_dataset 변수에 저장
print("요약 작업을 시작합니다...")
summarized_dataset = translated_dataset.map(summarize_english_lyrics)
print("요약 작업 완료.")

print("\n요약이 추가된 데이터셋 정보:")
print(summarized_dataset)

print("\n첫 번째 데이터의 영어 가사와 요약 비교:")
print(f"--- 곡: {summarized_dataset[0]['song_name']} / {summarized_dataset[0]['artist']} ---")
print("\n--- 영어 가사 (앞부분) ---")
print(summarized_dataset[0]['english_verse'][:300])
print("\n--- 영어 요약 (Summary) ---")
print(summarized_dataset[0]['summary'])

# 데이터셋 확인
df_check_summary = pd.DataFrame(summarized_dataset)
print("\n요약 추가 후 데이터셋 미리보기 (DataFrame):")
display(df_check_summary[['song_name', 'artist', 'genre', 'summary']].head(5))

요약 작업을 시작합니다...


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

요약 작업 완료.

요약이 추가된 데이터셋 정보:
Dataset({
    features: ['song_name', 'artist', 'genre', 'release_date', 'lyrics', 'english_verse', 'summary'],
    num_rows: 10
})

첫 번째 데이터의 영어 가사와 요약 비교:
--- 곡: FOREVER 1 / 소녀시대 (GIRLS' GENERATION) ---

--- 영어 가사 (앞부분) ---
FOREVER 1
It’s love It’s love
We’re not stopping
You're the only one in the world who can't stop me.
It's beautiful.
I was afraid to say, "I love you".
For eternity,
You and I
The tears that burst
You know what?
I just threw it all away.
I'm not hesitant to say anything.

--- 영어 요약 (Summary) ---
You're the only one in the world who can't stop me. It's beautiful. I was afraid to say, "I love you". For eternity, you and I.

요약 추가 후 데이터셋 미리보기 (DataFrame):


,song_name,artist,genre,summary
0,FOREVER 1,소녀시대 (GIRLS' GENERATION),댄스,You're the only one in the world who can't sto...
1,나야 나 (PICK ME),PRODUCE 101,댄스,"Remember, please, this moment tonight. Pick me..."
2,악몽,버블 시스터즈,댄스,I woke up in the morning with a nightmare and ...
3,Love,브라운아이드걸스,댄스,"My love for you, you little bitch. I need you...."
4,Young Gunz,신화,R&B/Soul,The Rock Dizzle C`MON is a song by South Korea...


In [ ]:
# @title 7. 감정 분석 모델 파이프라인 로드
"""
이 셀에서는 영어 텍스트의 감정을 분석하기 위한 모델 파이프라인을 로드함.
모델: SamLowe/roberta-base-go_emotions
파이프라인 타입: text-classification
28가지 세분화된 감정으로 분류
"""
# 'text-classification' 파이프라인을 로드하고, 사용할 모델은 'SamLowe/roberta-base-go_emotions'로 지정
emotion_classifier = pipeline(
    task="text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=1, # 가장 확률이 높은 한개
    device=device
)

print("감정 분석 파이프라인 로드 완료.")

# 감정 분석 테스트
test_emotion = emotion_classifier("I am very happy today!")
print(f"감정 분석 테스트: {test_emotion}")

감정 분석 파이프라인 로드 완료.
감정 분석 테스트: [[{'label': 'joy', 'score': 0.8936862349510193}]]


In [ ]:
# @title 8. 영어 요약본 감정 분석 및 데이터셋에 추가
"""
이 셀에서는 'summary' 컬럼의 텍스트에 대해 감정 분석을 수행함.
map 함수를 사용하여 감정 분석 함수를 적용하고,
가장 확률이 높은 감정 레이블을 'emotion'이라는 새로운 컬럼에 저장함.
"""
# 감정 분석을 수행하는 함수 정의
def analyze_emotion(example):
    """데이터셋의 'summary'를 받아 감정 분석 결과를 반환하는 함수"""
    # emotion_classifier 파이프라인을 사용하여 example['summary']의 감정을 분석
    emotion_result = emotion_classifier(example['summary'])
    # top_k=1 이므로 결과는 [[{'label': '...', 'score': ...}]] 형태
    # 결과에서 가장 확률 높은 감정의 'label' 값과 'score' 값을 각각 저장.
    example['emotion'] = emotion_result[0][0]['label']
    example['emotion_score'] = float(emotion_result[0][0]['score'])
    return example

# summarized_dataset 데이터셋의 map 함수를 사용하여 위에서 정의한 analyze_emotion 함수를 적용
# 결과를 final_dataset 변수에 저장.
print("감정 분석 작업을 시작합니다...")
final_dataset = summarized_dataset.map(analyze_emotion)
print("감정 분석 작업 완료.")

print("\n최종 데이터셋 정보:")
print(final_dataset)

print("\n첫 번째 데이터의 요약(summary)과 감정(emotion):")
print("--- 영어 요약 (Summary) ---")
print(final_dataset[0]['summary'])
print("\n--- 분석된 감정 (Emotion) ---")
print(f"{final_dataset[0]['emotion']} (score: {final_dataset[0]['emotion_score']:.3f})")

# 최종 데이터셋 확인
df_final = pd.DataFrame(final_dataset)
print("\n최종 데이터셋 미리보기 (DataFrame):")
display(df_final[['song_name', 'artist', 'genre', 'summary', 'emotion']].head(10))

감정 분석 작업을 시작합니다...


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

감정 분석 작업 완료.

최종 데이터셋 정보:
Dataset({
    features: ['song_name', 'artist', 'genre', 'release_date', 'lyrics', 'english_verse', 'summary', 'emotion', 'emotion_score'],
    num_rows: 10
})

첫 번째 데이터의 요약(summary)과 감정(emotion):
--- 영어 요약 (Summary) ---
You're the only one in the world who can't stop me. It's beautiful. I was afraid to say, "I love you". For eternity, you and I.

--- 분석된 감정 (Emotion) ---
admiration (score: 0.707)

최종 데이터셋 미리보기 (DataFrame):


,song_name,artist,genre,summary,emotion
0,FOREVER 1,소녀시대 (GIRLS' GENERATION),댄스,You're the only one in the world who can't sto...,admiration
1,나야 나 (PICK ME),PRODUCE 101,댄스,"Remember, please, this moment tonight. Pick me...",neutral
2,악몽,버블 시스터즈,댄스,I woke up in the morning with a nightmare and ...,excitement
3,Love,브라운아이드걸스,댄스,"My love for you, you little bitch. I need you....",love
4,Young Gunz,신화,R&B/Soul,The Rock Dizzle C`MON is a song by South Korea...,neutral
5,아저씨SWAG (Feat. 개코 of Dynamic Duo),싸이 (PSY),댄스,Even if we try to hide the traces of the years...,neutral
6,새 사랑,송하예,발라드,He's a man with a gun. He's been watching me f...,sadness
7,180도,벤,발라드,It's 180 degrees now. Love is all the same. Yo...,love
8,Shall We Dance,블락비 (Block B),댄스,You're going to be a little bit of a bitch. Yo...,annoyance
9,Counting Stars (Feat. Beenzino),BE'O (비오),랩/힙합,The night sky is full of pearls. Counting star...,neutral


In [ ]:
# @title 9. 결과 정리 및 마무리
"""
모든 단계를 거쳐 생성된 최종 데이터셋에는 원본 K-pop 가사 데이터에 'english_lyrics', 'summary', 'emotion', 'emotion_score' 컬럼이 추가됨
파이프라인 흐름: 한국어 가사 → 영어 번역 → 영어 요약 → 감정 분석
"""
print("모든 작업이 완료되었습니다.")
print("최종 데이터셋 컬럼:", final_dataset.column_names)

# 최종 결과 확인
for i in range(min(5, len(final_dataset))):
    print(f"\n--- 샘플 {i+1} ---")
    print(f"곡명: {final_dataset[i]['song_name']} / 아티스트: {final_dataset[i]['artist']} / 장르: {final_dataset[i]['genre']}")
    print(f"가사 일부: {final_dataset[i]['lyrics'][:80]}...")
    print(f"영어 번역 일부: {final_dataset[i]['english_verse'][:80]}...")
    print(f"영어 요약: {final_dataset[i]['summary']}")
    print(f"감정 분석: {final_dataset[i]['emotion']} (score: {final_dataset[i]['emotion_score']:.3f})")

# 필요시 CSV 저장
df_final.to_csv("kpop_lyrics_processed.csv", index=False, encoding='utf-8-sig')
print("\n결과를 'kpop_lyrics_processed.csv' 파일로 저장했습니다.")

모든 작업이 완료되었습니다.
최종 데이터셋 컬럼: ['song_name', 'artist', 'genre', 'release_date', 'lyrics', 'english_verse', 'summary', 'emotion', 'emotion_score']

--- 샘플 1 ---
곡명: FOREVER 1 / 아티스트: 소녀시대 (GIRLS' GENERATION) / 장르: 댄스
가사 일부: FOREVER 1
It’s love It’s love
We’re not stopping
네가 머문 이 세상이 더
아름다운 건
겁 없이 외치던 말...
영어 번역 일부: FOREVER 1
It’s love It’s love
We’re not stopping
You're the only one in the worl...
영어 요약: You're the only one in the world who can't stop me. It's beautiful. I was afraid to say, "I love you". For eternity, you and I.
감정 분석: admiration (score: 0.707)

--- 샘플 2 ---
곡명: 나야 나 (PICK ME) / 아티스트: PRODUCE 101 / 장르: 댄스
가사 일부: 너를 보던 그 순간
Pick me Pick me Pick me
시선 고정 너에게
눈부셔 Shining Shining
제발 내 맘을 Pick me...
영어 번역 일부: The moment I saw you.
Pick me Pick me Pick me
The line of sight fixed you.
Look ...
영어 요약: Remember, please, this moment tonight. Pick me up to the end. The moment I saw you. The line of sight fixed you.
감정 분석: neutral (score: 0.770)

--- 샘플 3 ---
곡명: 악몽 / 아티스트: 버블 시스터즈 